# AML_02 — Une validation qui mérite notre confiance

**Compétition de travail : [fraud](https://www.kaggle.com/competitions/playground-series-s6e11)**

**Question : Comment détecter un score trop beau pour être vrai ?**

7 heures pédagogiques, hors pauses. Les exemples sont des jalons techniques courts ; l’essentiel du temps est consacré aux exercices, revues et expériences. **Version étudiant.**

**Extension Kaggle guidée :** cellules 32 à 42. Comparaison, choix, audit, soumission CSV contrôlée et journal. Les dépôts Kaggle restent manuels, après lecture des règles de la compétition.

## Le cours et le notebook avancent ensemble

Ouvrir le [cours rédigé J2](https://github.com/MathieuDWeill/ai-course-student/blob/main/usecase/MyLittleMLLab/lectures/AML_02_cours.md) et la [fiche de séance](https://github.com/MathieuDWeill/ai-course-student/blob/main/usecase/MyLittleMLLab/fiches/AML_02.md). Le professeur montre un exemple, vous prévoyez une sortie, vous exécutez la cellule indiquée, puis nous corrigeons ensemble. Les numéros du PowerPoint comptent **toutes les cellules**, textes compris, et non les compteurs `In [n]`.

Nous réutilisons l’environnement et les données de J1. Une nouvelle installation ou un nouveau téléchargement Kaggle ne sont pas nécessaires si J1 fonctionne.

## Parcours guidé — Séparer apprendre, choisir et vérifier

**Objectif minimal :** Expliquer sur son propre jeu ce qui pourrait traverser le split. Vérifier les IDs avant de calculer un score.

**Idée à retenir :** Une évaluation devient trompeuse si la réponse ou une observation apparentée entre dans l’apprentissage.

Nous avançons en cinq étapes : **je montre → nous faisons ensemble → vous reproduisez → vous changez une chose → vous expliquez**. Ne pas lancer les autres cellules avant le signal de l’enseignant. Le premier exemple est volontairement minuscule et distinct des données de compétition.

### Exemple au tableau

Dessiner trois boîtes : TRAIN → VALIDATION → TEST FINAL. Écrire sous chaque boîte : apprendre / choisir / vérifier une fois les choix fixés.

Écrire des observations A1, A2, A3, B1, B2, C1, C2, D1. Le groupe A correspond à une même source, pas à trois sources indépendantes.

Proposer TRAIN={A1,A2,B1,C1}, VALIDATION={A3,B2,C2,D1}. Entourer A, B et C de la même couleur de chaque côté. Demander : « Est-ce que nous évaluons vraiment sur de nouvelles sources ? »

Exemple temporel : origine juin, cible connue en août ; validation qui démarre en juillet. Dessiner les deux dates : une origine ancienne ne rend pas son label disponible plus tôt.

### Répondez avant de lancer Python

1. Quel ensemble sert à apprendre le scaler ?
2. Pourquoi A1 et A3 peuvent-ils contaminer notre évaluation ?
3. Une ligne de juin avec un résultat connu en août peut-elle servir à décider en juillet ?
4. Que signifie une prédiction hors fold ?

## Les mots utilisés aujourd’hui

**Split** : séparation des observations en ensembles. **ID** : identifiant d’une observation. **Groupe** : source commune à plusieurs observations. **Fuite** : information indûment utilisée pour apprendre ou choisir. **Fold** : partie du découpage de validation croisée. **OOF** : prédiction produite pour une ligne par un modèle qui n’a pas utilisé cette ligne pour apprendre.

Le train local apprend. La validation locale aide à comparer. Le test officiel Kaggle ne contient pas ses réponses et reste hors des évaluations locales. À la fin de la séance, nous prédisons ses lignes pour préparer la soumission. La validation croisée porte uniquement sur le train local `tr`. Le fichier appelé `test_score` par scikit-learn contient ici les scores des folds de validation, pas ceux du test Kaggle.

Nous utilisons un split stratifié de développement pour la fraude. Il faudra le réexaminer si la situation visée exige de séparer des personnes, des événements ou des périodes. Les exemples de groupes et de dates sont pédagogiques, distincts des données de fraude.

In [ ]:
from pathlib import Path
import sys,os,json,time
import numpy as np,pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
HERE=Path.cwd()
ROOT=next((p if (p/'mllab/competition.py').exists() else p/'student') for p in [HERE,*HERE.parents] if (p/'mllab/competition.py').exists() or (p/'student/mllab/competition.py').exists())
sys.path.insert(0,str(ROOT))
from mllab.competition import SPECS,read_competition,protocol,run_competition,binary_scores,get_folder
KEY='fraud';SPEC=SPECS[KEY]
QUICK=os.environ.get('MLLAB_QUICK','0')=='1'
LIMIT=int(os.environ.get('MLLAB_LIMIT','1000' if QUICK else '15000'))
IMAGE_LIMIT=int(os.environ.get('MLLAB_IMAGE_LIMIT','24' if QUICK else '240'))
plt.rcParams.update({'figure.figsize':(10,4),'axes.spines.top':False,'axes.spines.right':False})
print('Compétition :',SPEC['slug'],'métrique :',SPEC['metric'],'sens :',SPEC['direction'])
if KEY!='catsdogs':a,test,sample,protocol_record=protocol(KEY)
else:
 from mllab.vision_competition import image_protocol
 image_protocol(limit=IMAGE_LIMIT)
print('Données officielles trouvées. Profil rapide :',QUICK,'; aucun score Kaggle inventé.')

### Retrouver le tableau en Python

Lire les entrées ; prévoir la sortie ; exécuter ; comparer avec le calcul manuel.

In [ ]:
# Exemple pédagogique : des lignes distinctes peuvent partager une source.
toy = pd.DataFrame({
    'observation': ['A1','A2','A3','B1','B2','C1','C2','D1'],
    'groupe': ['A','A','A','B','B','C','C','D'],
    'ensemble': ['train','train','validation','train','validation','train','validation','validation']
})
display(toy)
train_groups = {'A','B','C'}
valid_groups = {'A','B','C','D'}
print('Groupes partagés :', sorted(train_groups & valid_groups))
new_train = {'A','B','C'}
new_valid = {'D'}
assert not new_train & new_valid
print('Groupes partagés après séparation :', sorted(new_train & new_valid))
print('Si nous visons de nouvelles sources, une source ne doit pas traverser le split.')

## Figer et auditer le protocole

**Point d’arrêt avant le calcul 1.**

1. L’enseignant annonce la question traitée et montre les entrées utilisées.
2. Vous dites ce qui est appris sur le train et ce qui est seulement mesuré.
3. Vous exécutez sans changer le code ; vous repérez une sortie et son unité.
4. Vous expliquez ce que cette sortie permet de conclure et ce qu’elle ne prouve pas.

**Ne pas enchaîner si vous ne pouvez pas nommer la sortie.** Le professeur fait un exemple analogue ; les plus avancés ont une extension facultative.

In [ ]:
from mllab.competition import subsets
tr, va, _, _, protocol_record = subsets(KEY, limit=LIMIT)
shared_ids = set(tr.id) & set(va.id)
assert not shared_ids
print('Identifiants partagés :', len(shared_ids))
print('Train local :', len(tr), '; validation locale :', len(va))
balance = pd.DataFrame({
    'train_proportion': tr.is_fraud.value_counts(normalize=True),
    'validation_proportion': va.is_fraud.value_counts(normalize=True)
}).sort_index()
display(balance)
RUN_J2 = ROOT/'runs'/('j2_' + str(time.time_ns()))
RUN_J2.mkdir(parents=True)
print('Dossier de preuves créé :', RUN_J2.name)
print('Zéro ID partagé ne prouve pas l’absence de groupes ou d’informations futures.')

## Validation croisée à l’intérieur du train de développement

**Point d’arrêt avant le calcul 2.**

1. L’enseignant annonce la question traitée et montre les entrées utilisées.
2. Vous dites ce qui est appris sur le train et ce qui est seulement mesuré.
3. Vous exécutez sans changer le code ; vous repérez une sortie et son unité.
4. Vous expliquez ce que cette sortie permet de conclure et ce qu’elle ne prouve pas.

**Ne pas enchaîner si vous ne pouvez pas nommer la sortie.** Le professeur fait un exemple analogue ; les plus avancés ont une extension facultative.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from mllab.competition import make_model, xy
X = xy(tr, KEY)
y = tr.is_fraud
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=2026)
model = make_model(KEY, tr, 'linear')
folds = cross_validate(model, X, y, cv=cv, scoring='roc_auc')
fold_id = np.zeros(len(tr), dtype=int)
rows = []
for k, (i_train, i_valid) in enumerate(cv.split(X, y), start=1):
    assert not set(tr.iloc[i_train].id) & set(tr.iloc[i_valid].id)
    fold_id[i_valid] = k
    rows.append({'fold': k, 'n_train': len(i_train), 'n_validation': len(i_valid),
                 'fraudes_validation': int(y.iloc[i_valid].sum()),
                 'roc_auc': float(folds['test_score'][k-1])})
fold_table = pd.DataFrame(rows)
display(fold_table)
print('AUC moyenne des folds :', folds['test_score'].mean())
print('Écart-type descriptif :', folds['test_score'].std())
oof = cross_val_predict(model, X, y, cv=cv, method='predict_proba')[:, 1]
assert len(oof) == len(tr) and np.isfinite(oof).all()
print('Nombre de prédictions OOF :', len(oof))
print('AUC OOF regroupée :', binary_scores(y.to_numpy(), oof)['roc_auc'])
print('Les deux fonctions effectuent chacune 3 apprentissages, soit 6 ici.')
print('Les transformations sont réapprises dans chaque fold grâce au pipeline.')

## Même AUC, décision différente

**Point d’arrêt avant le calcul 3.**

1. L’enseignant annonce la question traitée et montre les entrées utilisées.
2. Vous dites ce qui est appris sur le train et ce qui est seulement mesuré.
3. Vous exécutez sans changer le code ; vous repérez une sortie et son unité.
4. Vous expliquez ce que cette sortie permet de conclure et ce qu’elle ne prouve pas.

**Ne pas enchaîner si vous ne pouvez pas nommer la sortie.** Le professeur fait un exemple analogue ; les plus avancés ont une extension facultative.

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score
rows = []
for threshold in [0.1, 0.5, 0.9]:
    decision = oof >= threshold
    tn, fp, fn, tp = confusion_matrix(y, decision, labels=[0,1]).ravel()
    scores = binary_scores(y.to_numpy(), oof, threshold)
    rows.append({'seuil': threshold, 'alertes': int(decision.sum()),
                 'vraies_alertes': int(tp), 'fausses_alertes': int(fp), 'fraudes_ratees': int(fn),
                 'precision': precision_score(y, decision, zero_division=0),
                 'rappel': recall_score(y, decision, zero_division=0),
                 'f1': scores['f1'], 'roc_auc': scores['roc_auc']})
threshold_table = pd.DataFrame(rows)
display(threshold_table)
print('Même oof : changer le seuil ne réentraîne pas et ne change pas la ROC-AUC.')

## Démonstrations accompagnées après le socle

Les cellules 22, 24 et 26 reprennent respectivement le prétraitement, les groupes et les dates sur des tableaux minuscules. Le professeur les montre et les commente avant toute modification.

La cellule 28 reprend le réglage C de J1 avec une validation croisée. Elle reste facultative si le socle n’est pas compris. La cellule 30 montre une fausse évaluation en utilisant volontairement la réponse comme score. **Ce n’est pas une méthode de prédiction utilisable.**

On ne recherche pas une amélioration au hasard. On écrit ce qui change, ce qui reste fixe et ce que la mesure permet de conclure.

**Point d’arrêt avant le calcul 4.**

1. L’enseignant annonce la question traitée et montre les entrées utilisées.
2. Vous dites ce qui est appris sur le train et ce qui est seulement mesuré.
3. Vous exécutez sans changer le code ; vous repérez une sortie et son unité.
4. Vous expliquez ce que cette sortie permet de conclure et ce qu’elle ne prouve pas.

**Ne pas enchaîner si vous ne pouvez pas nommer la sortie.** Le professeur fait un exemple analogue ; les plus avancés ont une extension facultative.

In [ ]:
from mllab.core import digest, source_hash, versions
proof = pd.DataFrame({'id': tr.id.to_numpy(), 'target': y.to_numpy(),
                      'fold_exclu': fold_id, 'probability_oof': oof})
assert not proof.id.duplicated().any()
assert (proof.fold_exclu > 0).all()
proof.to_csv(RUN_J2/'oof_predictions.csv', index=False)
fold_table.to_csv(RUN_J2/'fold_scores.csv', index=False)
threshold_table.to_csv(RUN_J2/'thresholds.csv', index=False)
manifest = {'question': 'Évaluer sans fuite sur le train local', 'seed': 2026,
            'limit': LIMIT, 'n_train_local': len(tr), 'n_validation_locale': len(va),
            'cv': 'StratifiedKFold(3, shuffle=True)', 'model': 'linear',
            'donnees_sha256': protocol_record['data_sha256'],
            'code_sha256': source_hash(), 'packages': versions(),
            'fichiers_sha256': {name: digest(RUN_J2/name) for name in
                               ['oof_predictions.csv','fold_scores.csv','thresholds.csv'] +
                               (['regularisation_cv.csv'] if (RUN_J2/'regularisation_cv.csv').exists() else [])}}
(RUN_J2/'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
display(proof.head())
print('Fichiers conservés :', ', '.join(sorted(p.name for p in RUN_J2.iterdir())))
HYPOTHESE = 'À compléter avec la question de votre essai'
CONCLUSION = 'À compléter avec une mesure, le protocole et une limite'
# Écrire ces variables ne change pas automatiquement le manifeste enregistré.

## Votre conclusion et les preuves

Dans `RUN_J2`, retrouvez `oof_predictions.csv`, `fold_scores.csv`, `thresholds.csv` et `manifest.json`. Les fichiers décrivent la validation croisée sur le train local, pas une soumission Kaggle.

Dans la cellule 19, complétez HYPOTHESE et CONCLUSION puis écrivez votre conclusion dans votre compte rendu. Si vous exécutez les extensions, la cellule 28 ajoute un tableau de comparaison de C au dossier.

**Questions de sortie :** pourquoi le scaler doit-il être appris dans chaque sous-train ? Que garantit une prédiction OOF, et que ne garantit-elle pas ? Quel split convient à l’usage que vous visez ?

## Cellule 22 — Une moyenne calculée au mauvais endroit

Exemple artificiel. Nous étudions l’effet du prétraitement, pas un score du concours. Prévoyez la moyenne sur le train, puis celle qui inclurait indûment la validation.

In [ ]:
from sklearn.preprocessing import StandardScaler
small_train = np.array([[10.0], [20.0], [30.0]])
small_valid = np.array([[1000.0]])
proper_scaler = StandardScaler().fit(small_train)
leaky_scaler = StandardScaler().fit(np.vstack([small_train, small_valid]))
print('Moyenne apprise sur le train seulement :', proper_scaler.mean_[0])
print('Moyenne contaminée par la validation :', leaky_scaler.mean_[0])
print('Valeur 20 transformée correctement :', proper_scaler.transform([[20.0]])[0,0])
print('Le pipeline réapprend ce traitement dans chaque fold.')

## Cellule 24 — Vérifier une séparation par groupe

Les observations A1, A2 et A3 viennent de la même source A. Nous vérifions les sources présentes dans les deux ensembles. Ce mini-tableau ne sert pas à mesurer une AUC.

In [ ]:
from sklearn.model_selection import GroupKFold
group_cv = GroupKFold(n_splits=2)
rows = []
for k, (i_train, i_valid) in enumerate(group_cv.split(toy, groups=toy.groupe), start=1):
    g_train = set(toy.iloc[i_train].groupe)
    g_valid = set(toy.iloc[i_valid].groupe)
    assert not g_train & g_valid
    rows.append({'fold': k, 'groupes_train': ', '.join(sorted(g_train)),
                 'groupes_validation': ', '.join(sorted(g_valid)),
                 'groupes_partages': len(g_train & g_valid)})
display(pd.DataFrame(rows))

## Cellule 26 — Une ligne ancienne avec une réponse encore inconnue

Exemple artificiel : nous décidons au 1er juillet. Une date d’origine ancienne ne suffit pas si la cible n’est connue qu’en août.

In [ ]:
timeline = pd.DataFrame({
    'id': ['P','Q','R','S'],
    'origine': pd.to_datetime(['2026-05-01','2026-06-01','2026-06-15','2026-04-01']),
    'cible_connue_le': pd.to_datetime(['2026-06-01','2026-08-01','2026-08-15','2026-05-01'])
})
cutoff = pd.Timestamp('2026-07-01')
timeline['utilisable_pour_apprendre'] = timeline.cible_connue_le <= cutoff
display(timeline)
print('Exemples disponibles au 1er juillet :', timeline.loc[timeline.utilisable_pour_apprendre,'id'].tolist())
print('Il faut aussi vérifier la disponibilité des entrées au moment de chaque prévision.')

## Cellule 28 — Le gain de régularisation se retrouve-t-il entre folds ?

Nouvelle expérience guidée. Même train local, mêmes folds, même métrique. C plus petit signifie une régularisation plus forte pour cette régression logistique. Nous comparons 0,1 et 1, sans toucher au test Kaggle.

In [ ]:
rows = []
for c in [0.1, 1.0]:
    candidate = make_model(KEY, tr, 'linear', params={'C': c})
    result = cross_validate(candidate, X, y, cv=cv, scoring='roc_auc')
    scores = result['test_score']
    rows.append({'C': c, 'fold_1': scores[0], 'fold_2': scores[1], 'fold_3': scores[2],
                 'auc_moyenne': scores.mean(), 'ecart_type_descriptif': scores.std()})
c_table = pd.DataFrame(rows)
display(c_table)
c_table.to_csv(RUN_J2/'regularisation_cv.csv', index=False)
print('Dispersion descriptive : ni test de significativité, ni intervalle de confiance.')
manifest_path = RUN_J2/'manifest.json'
if manifest_path.exists():
    saved_manifest = json.loads(manifest_path.read_text())
    saved_manifest['fichiers_sha256']['regularisation_cv.csv'] = digest(RUN_J2/'regularisation_cv.csv')
    manifest_path.write_text(json.dumps(saved_manifest, indent=2), encoding='utf-8')

## Cellule 30 — Un score parfait qui ne prédit rien

Démonstration volontairement invalide. Nous remplaçons les prévisions par la réponse connue pour montrer qu’un score seul ne valide pas une expérience.

In [ ]:
copie_de_la_reponse = y.to_numpy().copy()
print(binary_scores(y.to_numpy(), copie_de_la_reponse))
print('ÉVALUATION INVALIDE : le score a été fabriqué avec la cible connue.')
print('Pour une nouvelle transaction sans cible, ce calcul est impossible.')

## Le tournoi Kaggle local

Après les cellules 5, 10 et 13. Trois candidats, les mêmes données et trois folds identiques. Prévoyez la ROC-AUC d’un modèle qui donne le même score à chaque paiement. Le temps mesuré dépend de votre ordinateur.

In [ ]:
from time import perf_counter
candidates = [('dummy', {}), ('linear', {'C': 0.1}),
              ('boosting', {'max_iter': 80, 'max_leaf_nodes': 15})]
comparison_rows = []
for name, params in candidates:
    start = perf_counter()
    result = cross_validate(make_model(KEY, tr, name, params=params), X, y,
                            cv=cv, scoring='roc_auc')
    values = result['test_score']
    comparison_rows.append({'modele': name, 'fold_1': values[0], 'fold_2': values[1],
        'fold_3': values[2], 'auc_moyenne': values.mean(),
        'ecart_type': values.std(), 'secondes': round(perf_counter()-start, 2)})
comparison = pd.DataFrame(comparison_rows)
display(comparison.round(6))
comparison.to_csv(RUN_J2/'comparaison_modeles.csv', index=False)
print('9 apprentissages. Temps local, sans garantie de gain sur Kaggle.')

## Le choix avant la validation locale

Après la cellule 32. Nous sélectionnons sur les folds du train local. Nous ne consulterons la validation locale qu’après ce choix. Le boosting regroupe des petits arbres successifs. Son encodage ordinal traite ici les catégories comme des nombres, ce qui constitue une limite.

In [ ]:
winner = comparison.sort_values('auc_moyenne', ascending=False).iloc[0]
SELECTED_MODEL = str(winner.modele)
SELECTED_PARAMS = dict(next(params for name, params in candidates if name == SELECTED_MODEL))
selection = {'modele': SELECTED_MODEL, 'params': SELECTED_PARAMS,
             'critere': 'ROC-AUC moyenne sur les 3 folds du train local',
             'auc_cv': float(winner.auc_moyenne), 'train_limit': LIMIT}
(RUN_J2/'selection.json').write_text(json.dumps(selection, indent=2), encoding='utf-8')
print(json.dumps(selection, indent=2))
print('Choix enregistré avant de regarder la validation locale.')

## Une vérification séparée après le choix

Après la cellule 34. Cette validation locale est un contrôle de développement, pas une garantie finale. Si on adapte nos choix après sa lecture, elle participe à la sélection et ne reste plus indépendante.

In [ ]:
selected = make_model(KEY, tr, SELECTED_MODEL, params=SELECTED_PARAMS)
selected.fit(X, y)
p_valid = selected.predict_proba(xy(va, KEY))[:, 1]
valid_metrics = binary_scores(va.is_fraud.to_numpy(), p_valid)
display(pd.DataFrame([{'modele': SELECTED_MODEL, 'n_validation': len(va),
    'auc_cv': selection['auc_cv'], 'auc_validation_locale': valid_metrics['roc_auc']}]).round(6))
(RUN_J2/'controle_validation.json').write_text(json.dumps(valid_metrics, indent=2))
print('Aucun score du leaderboard ne peut être déduit de ces valeurs.')

## Les erreurs du modèle retenu

Après la cellule 36. Le seuil de 0,5 sert ici à illustrer les erreurs. Pour notre soumission ROC-AUC, nous conservons les scores continus. Décrivez un faux négatif et un faux positif sans inventer une explication causale.

In [ ]:
audit = pd.DataFrame({'id': va.id.to_numpy(), 'cible': va.is_fraud.to_numpy(),
                      'score': p_valid, 'alerte_05': (p_valid >= 0.5).astype(int)})
audit['erreur'] = np.where((audit.cible == 1) & (audit.alerte_05 == 0), 'retard_detection',
                  np.where((audit.cible == 0) & (audit.alerte_05 == 1), 'fausse_alerte', 'correct'))
display(audit.groupby('erreur').size().rename('nombre').reset_index())
display(audit[audit.erreur != 'correct'].sort_values('score', ascending=False).head(5).round(6))
audit.to_csv(RUN_J2/'audit_validation.csv', index=False)
print('retard_detection désigne ici une fraude manquée au seuil 0,5.')

## Le fichier de soumission Kaggle

Après la cellule 36. Nous réapprenons le modèle choisi sur un échantillon stratifié du train officiel, de taille LIMIT, puis prédisons tout le test officiel. C’est une version CPU rapide. Le test officiel n’a pas de labels. Cette cellule ne transmet rien à Kaggle.

In [ ]:
from mllab.competition import make_submission
SUBMISSION_DIR = make_submission(KEY, model=SELECTED_MODEL,
    params=SELECTED_PARAMS, seed=2026, limit=LIMIT)
submission = pd.read_csv(SUBMISSION_DIR/'submission.csv')
assert list(submission.columns) == list(sample.columns)
assert len(submission) == len(test)
assert np.array_equal(submission.id.to_numpy(), sample.id.to_numpy())
assert submission.id.is_unique
assert np.isfinite(submission.is_fraud.to_numpy()).all()
assert submission.is_fraud.between(0, 1).all()
display(submission.head().round(6))
print('Lignes du test officiel :', len(test))
print('Lignes du CSV :', len(submission))
print('Colonnes :', ', '.join(submission.columns))
print('Contrôles du format : OK. Aucun envoi automatique.')
print('Fichier à retrouver :', SUBMISSION_DIR/'submission.csv')

## Le journal des soumissions

Après la cellule 40. Après un dépôt manuel accepté par Kaggle, remplacez None par votre score réellement affiché puis relancez cette cellule. Ne recopiez pas l’AUC locale dans la colonne Kaggle. Chaque exécution remplace la ligne de cette expérience, sans créer de doublon.

In [ ]:
KAGGLE_PUBLIC_SCORE = None  # Remplacer seulement après retour réel de Kaggle
record = {'experience': RUN_J2.name, 'modele': SELECTED_MODEL,
          'auc_cv': selection['auc_cv'], 'auc_validation_locale': valid_metrics['roc_auc'],
          'score_public_kaggle': KAGGLE_PUBLIC_SCORE,
          'submission_csv': str(SUBMISSION_DIR/'submission.csv'),
          'limite_train': LIMIT, 'decision_suivante': 'À expliquer avec une hypothèse'}
pd.DataFrame([record]).to_csv(RUN_J2/'journal_kaggle.csv', index=False)
print('Modèle :', SELECTED_MODEL)
print('Score public Kaggle :', 'non renseigné' if KAGGLE_PUBLIC_SCORE is None else KAGGLE_PUBLIC_SCORE)
print('Journal écrit : journal_kaggle.csv')
print('Prochaine expérience : une seule modification justifiée, sur les mêmes folds.')